In [1]:
import json
file_path = "dataset/authors.txt"
with open(file_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        # Cấu trúc Open Library phân tách các cột bằng dấu Tab (\t)
        parts = line.strip().split("\t")
        
        # Kiểm tra xem dòng đó có đủ cấu trúc 5 cột hay không
        if len(parts) >= 5:
            record_key = parts[1]  # Mã định danh (Ví dụ: /books/OL1M)
            json_str = parts[4]    # Chuỗi JSON thô nằm ở cột thứ 5 (index 4)
            
            try:
                # Chuyển chuỗi text thành Python Dictionary (để xử lý dữ liệu)
                data_dict = json.loads(json_str)
                
                # --- KHÚC NÀY LÀ NƠI BẠN XỬ LÝ HOẶC ĐẨY VÀO DATABASE (JSONB) ---
                # Ví dụ: In thử 3 dòng đầu tiên để xem cấu trúc
                if line_num <= 3:
                    print(f"\n[Dòng {line_num}] Mã sách: {record_key}")
                    print(f"Tiêu đề: {data_dict.get('title')}")
                else:
                    break  # Bỏ dòng này khi bạn muốn chạy toàn bộ file
                    
            except json.JSONDecodeError:
                print(f"Bỏ qua dòng {line_num} do lỗi định dạng JSON")
                continue

print("Hoàn thành!")


[Dòng 1] Mã sách: /authors/OL10000080A
Tiêu đề: None

[Dòng 2] Mã sách: /authors/OL10000104A
Tiêu đề: None

[Dòng 3] Mã sách: /authors/OL10000674A
Tiêu đề: None
Hoàn thành!


In [2]:
import json

# Đường dẫn tới file .txt của bạn
file_path = "dataset/authors.txt"

# Số lượng dòng bạn muốn quét để dò schema (tăng lên nếu muốn chính xác hơn)
sample_size = 20000

# Bộ tập hợp để lưu tất cả các keys tìm thấy (set giúp tự động loại bỏ trùng lặp)
all_schemas_keys = set()


def extract_keys(data, prefix=""):
    """Hàm đệ quy để lấy toàn bộ schema, kể cả các trường lồng nhau ẩn sâu"""
    keys = []
    if isinstance(data, dict):
        for k, v in data.items():
            key_name = f"{prefix}.{k}" if prefix else k
            keys.append(key_name)
            # Nếu giá trị bên trong lại là một Dictionary, tiếp tục đào sâu xuống
            if isinstance(v, dict):
                keys.extend(extract_keys(v, key_name))
            # Nếu là danh sách các Dictionary, cũng đào sâu xuống phần tử đầu tiên
            elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                keys.extend(extract_keys(v[0], f"{key_name}[]"))
    return keys


print(f"Đang quét thử {sample_size} dòng đầu tiên để dò tìm schema...")

with open(file_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        parts = line.strip().split("\t")
        if len(parts) >= 5:
            json_str = parts[4]
            try:
                data_dict = json.loads(json_str)

                # Lấy tất cả các khóa của dòng hiện tại và nạp vào bộ tập hợp tổng
                current_keys = extract_keys(data_dict)
                all_schemas_keys.update(current_keys)

            except json.JSONDecodeError:
                continue

        if line_num >= sample_size:
            break

# Sắp xếp lại danh sách các thuộc tính theo bảng chữ cái cho dễ nhìn
sorted_schema = sorted(list(all_schemas_keys))

print("\n=== KẾT QUẢ: TẤT CẢ CÁC TRƯỜNG (SCHEMA) TÌM THẤY ===")
for key in sorted_schema:
    print(f"- {key}")

print(f"\nTổng cộng tìm thấy: {len(sorted_schema)} thuộc tính khác nhau.")


Đang quét thử 20000 dòng đầu tiên để dò tìm schema...

=== KẾT QUẢ: TẤT CẢ CÁC TRƯỜNG (SCHEMA) TÌM THẤY ===
- alternate_names
- bio
- bio.type
- bio.value
- birth_date
- created
- created.type
- created.value
- date
- death_date
- entity_type
- key
- last_modified
- last_modified.type
- last_modified.value
- latest_revision
- links
- links[].title
- links[].type
- links[].type.key
- links[].url
- name
- personal_name
- photos
- remote_ids
- remote_ids.amazon
- remote_ids.bookbrainz
- remote_ids.gnd
- remote_ids.goodreads
- remote_ids.imdb
- remote_ids.isni
- remote_ids.lc_naf
- remote_ids.librarything
- remote_ids.librivox
- remote_ids.musicbrainz
- remote_ids.opac_sbn
- remote_ids.project_gutenberg
- remote_ids.storygraph
- remote_ids.viaf
- remote_ids.wikidata
- revision
- source_records
- title
- type
- type.key

Tổng cộng tìm thấy: 45 thuộc tính khác nhau.


In [3]:
import json

file_path = "dataset/authors.txt"
output_path = "sample_data.json"
samples = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 5:
            try:
                data = json.loads(parts[4])
                # Chọn những bản ghi có nhiều dữ liệu một chút để xem cấu trúc
                if len(data.keys()) > 8: 
                    samples.append(data)
                if len(samples) >= 5: # Lấy 5 mẫu là đủ để xem
                    break
            except:
                continue

# Ghi ra file JSON định dạng đẹp (thụt lề 4 dấu cách)
with open(output_path, "w", encoding="utf-8") as out_f:
    json.dump(samples, out_f, indent=4, ensure_ascii=False)

print(f"Đã tạo xong file trực quan tại: {output_path}")


Đã tạo xong file trực quan tại: sample_data.json
